# Sweeps: is the SFT baseline fair, and is KD an "anchoring dial"?

Needs GPU T4 x2, Internet, and the `HF_TOKEN` secret ticked. Trains + evaluates each job on the SAME
data and the SAME 121-task unseen-tool set as the main pipeline (see evaluation/sweep_worker.py).

Pick ONE experiment below (`EXPERIMENT`) per run -- each is sized to fit one Kaggle session:

| EXPERIMENT | what it answers | jobs | ~time |
|---|---|---|---|
| `sft_fair` | does a gentler SFT recipe (lr 5e-5, and lr 5e-5 + 1 epoch) remove SFT's seed instability? (main pair, 3 seeds) | 6, no teacher, 2 GPUs in parallel | ~2 h |
| `dial_main` | KD weight 0.2 / 0.8 with a teacher-free self anchor, strong student (main pair, 3 seeds); 0.5 already exists | 6, sequential | ~3.5 h |
| `dial_small_probe` | can LOWERING the anchor (KD weight 0.2 / 0.05) rescue the weak student? (small pair, seed 0) | 2, sequential | ~3.3 h |
| `dial_small_full` | same, remaining seeds 1 and 2 for the sweep you pick with `SMALL_SWEEP` | 2, sequential | ~3.3 h |
| `dial_main_vheavy` | the light end of the dial (KD weight 0.05) for the strong student, main pair, 3 seeds | 3, sequential | ~2 h |
| `dial_small_threshold` | KD weight 0.1 for the small student, 3 seeds - locates where learning collapses between 0.05 and 0.2 | 3, sequential | ~3 h |
| `data_scale` | does KD/anchoring matter more with LESS data? sft_only vs self_distill trained on 20 and 40 examples per tool (160 / 320 total; the full run has 80/tool = 640), main pair, 3 seeds | 12 (6 in parallel, then 6 sequential) | ~6 h |

Finished jobs are skipped (checked on Hugging Face), so a run can simply be repeated.

In [ ]:
EXPERIMENT = "sft_fair"          # sft_fair | dial_main | dial_small_probe | dial_small_full
SMALL_SWEEP = "sft_heavy"        # only for dial_small_full: the sweep (sft_heavy or sft_vheavy) that worked in the probe
MAX_MINUTES = 240                # per phase: stop starting new jobs after this long (data_scale needs 330, set below)

SEEDS = [0, 1, 2]
PRESETS = {
    "sft_fair": [f"{s}:main:{sw}:sft_only" for sw in ("lower_lr", "lower_lr_1ep") for s in SEEDS],
    "dial_main": [f"{s}:main:{sw}:self_distill" for sw in ("sft_heavy", "kd_heavy") for s in SEEDS],
    "dial_small_probe": [f"0:small:{sw}:self_distill_small" for sw in ("sft_heavy", "sft_vheavy")],
    "dial_small_full": [f"{s}:small:{SMALL_SWEEP}:self_distill_small" for s in (1, 2)],
    "dial_main_vheavy": [f"{s}:main:sft_vheavy:self_distill" for s in SEEDS],
    "dial_small_threshold": [f"{s}:small:kd_0p1:self_distill_small" for s in SEEDS],
    # condition-major so both GPU workers get a similar mix; @n<k> = k training examples per tool
    "data_scale": [f"{s}:main:baseline@n{k}:{c}" for c in ("sft_only", "self_distill") for k in (20, 40) for s in SEEDS],
}
if EXPERIMENT == "data_scale":
    MAX_MINUTES = 330
JOBS = PRESETS[EXPERIMENT]
print(EXPERIMENT, "->", len(JOBS), "jobs:", JOBS)

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Data (the main pipeline's own default split)

In [ ]:
run_module("adbench.data.prepare", "--config", "configs/data.yaml")

## 2. Check the token can write

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN is not set: tick it under Add-ons -> Secrets for this notebook, then run again.")
os.environ["ADBENCH_RUN_TAG"] = "v2-seed0"
from adbench import pipeline_state as ps
ps.check_upload()      # raises if the token cannot write

## 3. Train + evaluate

Same two-phase orchestration as notebooks 14/15: a job whose KD weight is > 0 loads a teacher (here the
student's own frozen base) next to the student, which only splits across both T4s when the process sees
both -- so those run sequentially in one process, while teacher-free jobs run as two GPU-pinned
workers in parallel.

In [ ]:
import subprocess
import sys
import threading

from adbench.training.train import load_experiment_config as _load_exp_cfg, resolve_training_config as _resolve_cfg

_exp_cfg = _load_exp_cfg("configs/experiment.yaml")


def needs_teacher(job):
    _, _, sweep, condition = job.split(":")
    sweep = sweep.split("@")[0]      # drop a data-scale suffix (@n20)
    return _resolve_cfg(_exp_cfg, condition, None if sweep == "baseline" else sweep).kd.kd_weight > 0


TEACHER_JOBS = [j for j in JOBS if needs_teacher(j)]
NO_TEACHER_JOBS = [j for j in JOBS if not needs_teacher(j)]
print("teacher jobs (sequential, both GPUs):", TEACHER_JOBS)
print("no-teacher jobs (parallel, one GPU each):", NO_TEACHER_JOBS)


def pump(proc, name):
    for line in proc.stdout:
        if "Loading weights" in line or "it/s]" in line:
            continue
        print(f"[{name}] {line.rstrip()}", flush=True)


def run_worker(jobs, env_overrides, tag):
    env = {**os.environ, **env_overrides, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [sys.executable, "-m", "adbench.evaluation.sweep_worker", "--jobs", ",".join(jobs),
         "--work-dir", f"/kaggle/working/sweep_{tag}", "--max-minutes", str(MAX_MINUTES)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
    )
    thread = threading.Thread(target=pump, args=(proc, tag), daemon=True)
    thread.start()
    return proc, thread


if NO_TEACHER_JOBS:
    print(f"### Phase A: {len(NO_TEACHER_JOBS)} no-teacher jobs, two parallel workers ###")
    workers = []
    for gpu in (0, 1):
        share = NO_TEACHER_JOBS[gpu::2]
        if share:
            workers.append(run_worker(share, {"CUDA_VISIBLE_DEVICES": str(gpu)}, f"phaseA_gpu{gpu}"))
    for proc, thread in workers:
        proc.wait()
        thread.join(timeout=30)
    codes = [proc.returncode for proc, _ in workers]
    print("Phase A exit codes:", codes)
    if any(codes):
        raise RuntimeError("a Phase A worker failed; see the log above")

if TEACHER_JOBS:
    print(f"### Phase B: {len(TEACHER_JOBS)} teacher jobs, sequential, both GPUs ###")
    proc, thread = run_worker(TEACHER_JOBS, {}, "phaseB")
    proc.wait()
    thread.join(timeout=30)
    print("Phase B exit code:", proc.returncode)
    if proc.returncode:
        raise RuntimeError("the Phase B worker failed; see the log above")

## Summary of what was evaluated

In [ ]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/sweep_phase*/results/*.json"):
    rows += json.load(open(path, encoding="utf-8"))
if rows:
    df = pd.DataFrame(rows)
    table = df.groupby(["pair", "sweep", "condition", "seed", "chain_length"]).success.mean().unstack("chain_length").round(3)
    print(table)
    print(df.groupby(["pair", "sweep", "condition", "seed"]).success.mean().round(3))
else:
    print("no result files in this session (everything was already on Hugging Face)")